In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import (
    DistilBertTokenizer,
    DistilBertForSequenceClassification,
    get_linear_schedule_with_warmup
)
from torch.optim import AdamW
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    roc_curve,
    auc
)
from pathlib import Path
import json
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

# Set seeds
np.random.seed(42)
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)

print("✅ Libraries imported")
print(f"PyTorch version: {torch.__version__}")

c:\Users\Adhiraj\miniconda3\envs\AI\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ Libraries imported
PyTorch version: 2.12.0+cpu
CUDA available: False
⚠️  Running on CPU (training will be slower ~30-40 mins)


In [2]:
# Step 2: Set correct path (update with YOUR folder name from above)
project_dir = Path(r'C:\\Python\\ML Intro\\Projects\\Sentiment Analysis Dashboard')

print(f"Using: {project_dir}")
print(f"Exists: {project_dir.exists()}")

Using: C:\Python\ML Intro\Projects\Sentiment Analysis Dashboard
Exists: True


In [3]:
# Create config if it doesn't exist
config_path = project_dir / 'config.json'

if not config_path.exists():
    print("⚠️  config.json not found, creating it...")
    
    # Create all required directories
    for d in ['data', 'models', 'results/plots', 'results/metrics', 'notebooks']:
        (project_dir / d).mkdir(parents=True, exist_ok=True)
    
    # Create config
    config = {
        'project_dir': str(project_dir),
        'train_samples': 10000,
        'test_samples': 2000,
        'num_classes': 2,
        'classes': ['Negative', 'Positive'],
        'max_length': 256,
        'model_name': 'distilbert-base-uncased',
        'batch_size': 16,
        'epochs': 3,
        'learning_rate': 2e-5
    }
    
    with open(config_path, 'w') as f:
        json.dump(config, f, indent=4)
    
    print(f"✅ config.json created: {config_path}")
else:
    with open(config_path, 'r') as f:
        config = json.load(f)
    print(f"✅ config.json loaded from: {config_path}")

print(f"\n📝 Config:")
for k, v in config.items():
    print(f"   {k}: {v}")

✅ config.json loaded from: C:\Python\ML Intro\Projects\Sentiment Analysis Dashboard\config.json

📝 Config:
   project_dir: C:\Python\ML Intro\Projects\Sentiment Analysis Dashboard
   train_samples: 10000
   test_samples: 2000
   num_classes: 2
   classes: ['Negative', 'Positive']
   max_length: 256
   model_name: distilbert-base-uncased
   batch_size: 16
   epochs: 3
   learning_rate: 2e-05


In [4]:
# Check if preprocessed data exists
train_path = project_dir / 'data' / 'train_processed.csv'
test_path = project_dir / 'data' / 'test_processed.csv'

if train_path.exists() and test_path.exists():
    print("✅ Data files found!")
    train_df = pd.read_csv(train_path)
    test_df = pd.read_csv(test_path)
    print(f"   Train samples: {len(train_df)}")
    print(f"   Test samples: {len(test_df)}")
    
else:
    print("⚠️  Data files not found, re-downloading...")
    
    from datasets import load_dataset
    
    print("📥 Loading Amazon Reviews dataset...")
    dataset = load_dataset(
        "amazon_polarity",
        split={
            'train': 'train[:10000]',
            'test': 'test[:2000]'
        }
    )
    
    train_df = pd.DataFrame(dataset['train'])
    test_df = pd.DataFrame(dataset['test'])
    
    # Add required columns
    import re
    import nltk
    nltk.download('stopwords', quiet=True)
    
    def preprocess_text(text):
        text = str(text).lower()
        text = re.sub(r'<[^>]+>', '', text)
        text = re.sub(r'http\S+|www\S+', '', text)
        text = re.sub(r'[^a-zA-Z\s]', ' ', text)
        text = re.sub(r'\s+', ' ', text).strip()
        return text
    
    print("🔄 Preprocessing text...")
    train_df['cleaned_text'] = train_df['content'].apply(preprocess_text)
    test_df['cleaned_text'] = test_df['content'].apply(preprocess_text)
    train_df['text_length'] = train_df['content'].str.len()
    train_df['word_count'] = train_df['content'].str.split().str.len()
    test_df['text_length'] = test_df['content'].str.len()
    test_df['word_count'] = test_df['content'].str.split().str.len()
    
    # Save
    (project_dir / 'data').mkdir(parents=True, exist_ok=True)
    train_df.to_csv(train_path, index=False)
    test_df.to_csv(test_path, index=False)
    
    print(f"✅ Data downloaded and saved!")
    print(f"   Train samples: {len(train_df)}")
    print(f"   Test samples: {len(test_df)}")

✅ Data files found!
   Train samples: 10000
   Test samples: 2000


In [5]:
from sklearn.model_selection import train_test_split

# Split train into train and validation
train_data, val_data = train_test_split(
    train_df,
    test_size=0.15,
    random_state=42,
    stratify=train_df['label']
)

# Reset indices
train_data = train_data.reset_index(drop=True)
val_data = val_data.reset_index(drop=True)
test_data = test_df.reset_index(drop=True)

print("✅ Data split created:")
print(f"   Training samples: {len(train_data)}")
print(f"   Validation samples: {len(val_data)}")
print(f"   Test samples: {len(test_data)}")

# Verify label distribution
print(f"\n📊 Label Distribution:")
print(f"   Train - Positive: {(train_data['label']==1).sum()}, Negative: {(train_data['label']==0).sum()}")
print(f"   Val   - Positive: {(val_data['label']==1).sum()}, Negative: {(val_data['label']==0).sum()}")
print(f"   Test  - Positive: {(test_data['label']==1).sum()}, Negative: {(test_data['label']==0).sum()}")

✅ Data split created:
   Training samples: 8500
   Validation samples: 1500
   Test samples: 2000

📊 Label Distribution:
   Train - Positive: 4168, Negative: 4332
   Val   - Positive: 735, Negative: 765
   Test  - Positive: 1046, Negative: 954


In [6]:
from transformers import DistilBertTokenizer

print("🔄 Loading DistilBERT tokenizer...")

MODEL_NAME = 'distilbert-base-uncased'
tokenizer = DistilBertTokenizer.from_pretrained(MODEL_NAME)

print(f"✅ Tokenizer loaded: {MODEL_NAME}")
print(f"   Vocabulary size: {tokenizer.vocab_size:,}")

# Test tokenizer on sample text
sample_text = "This product is absolutely amazing! I love it."
tokens = tokenizer.tokenize(sample_text)
encoding = tokenizer(
    sample_text,
    max_length=64,
    padding='max_length',
    truncation=True,
    return_tensors='pt'
)

print(f"\n📝 Tokenizer Test:")
print(f"   Original: '{sample_text}'")
print(f"   Tokens: {tokens}")
print(f"   Input IDs shape: {encoding['input_ids'].shape}")
print(f"   Attention mask shape: {encoding['attention_mask'].shape}")

🔄 Loading DistilBERT tokenizer...


✅ Tokenizer loaded: distilbert-base-uncased
   Vocabulary size: 30,522

📝 Tokenizer Test:
   Original: 'This product is absolutely amazing! I love it.'
   Tokens: ['this', 'product', 'is', 'absolutely', 'amazing', '!', 'i', 'love', 'it', '.']
   Input IDs shape: torch.Size([1, 64])
   Attention mask shape: torch.Size([1, 64])


In [7]:
class SentimentDataset(Dataset):
    """
    PyTorch Dataset for sentiment analysis
    """
    
    def __init__(self, texts, labels, tokenizer, max_length=256):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length
    
    def __len__(self):
        return len(self.texts)
    
    def __getitem__(self, idx):
        text = str(self.texts[idx])
        label = int(self.labels[idx])
        
        # Tokenize text
        encoding = self.tokenizer(
            text,
            add_special_tokens=True,
            max_length=self.max_length,
            padding='max_length',
            truncation=True,
            return_attention_mask=True,
            return_tensors='pt'
        )
        
        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'label': torch.tensor(label, dtype=torch.long)
        }

# Create datasets
MAX_LENGTH = config['max_length']

train_dataset = SentimentDataset(
    texts=train_data['content'].values,
    labels=train_data['label'].values,
    tokenizer=tokenizer,
    max_length=MAX_LENGTH
)

val_dataset = SentimentDataset(
    texts=val_data['content'].values,
    labels=val_data['label'].values,
    tokenizer=tokenizer,
    max_length=MAX_LENGTH
)

test_dataset = SentimentDataset(
    texts=test_data['content'].values,
    labels=test_data['label'].values,
    tokenizer=tokenizer,
    max_length=MAX_LENGTH
)

print("✅ Datasets created:")
print(f"   Train: {len(train_dataset)} samples")
print(f"   Val: {len(val_dataset)} samples")
print(f"   Test: {len(test_dataset)} samples")

# Test dataset
sample = train_dataset[0]
print(f"\n📊 Sample batch:")
print(f"   input_ids shape: {sample['input_ids'].shape}")
print(f"   attention_mask shape: {sample['attention_mask'].shape}")
print(f"   label: {sample['label']}")

✅ Datasets created:
   Train: 8500 samples
   Val: 1500 samples
   Test: 2000 samples

📊 Sample batch:
   input_ids shape: torch.Size([256])
   attention_mask shape: torch.Size([256])
   label: 0


In [8]:
BATCH_SIZE = config['batch_size']

# Create DataLoaders
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,  # Set to 0 for Windows
    pin_memory=True if torch.cuda.is_available() else False
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=True if torch.cuda.is_available() else False
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=True if torch.cuda.is_available() else False
)

print("✅ DataLoaders created:")
print(f"   Train batches: {len(train_loader)}")
print(f"   Val batches: {len(val_loader)}")
print(f"   Test batches: {len(test_loader)}")
print(f"   Batch size: {BATCH_SIZE}")

# Test one batch
batch = next(iter(train_loader))
print(f"\n📊 Sample batch shapes:")
print(f"   input_ids: {batch['input_ids'].shape}")
print(f"   attention_mask: {batch['attention_mask'].shape}")
print(f"   labels: {batch['label'].shape}")

✅ DataLoaders created:
   Train batches: 532
   Val batches: 94
   Test batches: 125
   Batch size: 16

📊 Sample batch shapes:
   input_ids: torch.Size([16, 256])
   attention_mask: torch.Size([16, 256])
   labels: torch.Size([16])


In [9]:
from transformers import DistilBertForSequenceClassification

print("🔄 Loading DistilBERT model...")

# Load pre-trained model with classification head
model = DistilBertForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2  # Binary: Positive/Negative
)

# Move model to GPU/CPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)

print(f"✅ Model loaded: {MODEL_NAME}")
print(f"   Device: {device}")

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"\n📊 Model Parameters:")
print(f"   Total: {total_params:,}")
print(f"   Trainable: {trainable_params:,}")
print(f"   Non-trainable: {total_params - trainable_params:,}")

🔄 Loading DistilBERT model...


Loading weights: 100%|██████████| 100/100 [00:00<00:00, 1767.79it/s]
[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


✅ Model loaded: distilbert-base-uncased
   Device: cpu

📊 Model Parameters:
   Total: 66,955,010
   Trainable: 66,955,010
   Non-trainable: 0


In [10]:
from transformers import get_linear_schedule_with_warmup
from torch.optim import AdamW

EPOCHS = config['epochs']
LEARNING_RATE = config['learning_rate']

# Setup optimizer
optimizer = AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    eps=1e-8,
    weight_decay=0.01
)

# Total training steps
total_steps = len(train_loader) * EPOCHS

# Warmup steps (10% of total)
warmup_steps = total_steps // 10

# Learning rate scheduler
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps
)

print("✅ Optimizer and scheduler configured:")
print(f"   Optimizer: AdamW")
print(f"   Learning rate: {LEARNING_RATE}")
print(f"   Epochs: {EPOCHS}")
print(f"   Total steps: {total_steps}")
print(f"   Warmup steps: {warmup_steps}")
print(f"   Weight decay: 0.01")

✅ Optimizer and scheduler configured:
   Optimizer: AdamW
   Learning rate: 2e-05
   Epochs: 3
   Total steps: 1596
   Warmup steps: 159
   Weight decay: 0.01


In [11]:
def train_epoch(model, data_loader, optimizer, scheduler, device):
    """Train for one epoch"""
    
    model.train()
    total_loss = 0
    all_preds = []
    all_labels = []
    
    for batch in tqdm(data_loader, desc='Training'):
        # Move to device
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['label'].to(device)
        
        # Zero gradients
        optimizer.zero_grad()
        
        # Forward pass
        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels
        )
        
        loss = outputs.loss
        logits = outputs.logits
        
        # Backward pass
        loss.backward()
        
        # Clip gradients
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        
        # Update weights
        optimizer.step()
        scheduler.step()
        
        # Track metrics
        total_loss += loss.item()
        preds = torch.argmax(logits, dim=1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
    
    avg_loss = total_loss / len(data_loader)
    accuracy = accuracy_score(all_labels, all_preds)
    
    return avg_loss, accuracy


def evaluate(model, data_loader, device):
    """Evaluate model on data"""
    
    model.eval()
    total_loss = 0
    all_preds = []
    all_labels = []
    all_probs = []
    
    with torch.no_grad():
        for batch in tqdm(data_loader, desc='Evaluating'):
            # Move to device
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['label'].to(device)
            
            # Forward pass
            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                labels=labels
            )
            
            loss = outputs.loss
            logits = outputs.logits
            
            # Get predictions and probabilities
            probs = torch.softmax(logits, dim=1)
            preds = torch.argmax(logits, dim=1)
            
            total_loss += loss.item()
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            all_probs.extend(probs.cpu().numpy())
    
    avg_loss = total_loss / len(data_loader)
    accuracy = accuracy_score(all_labels, all_preds)
    
    return avg_loss, accuracy, all_preds, all_labels, all_probs

print("✅ Training and evaluation functions defined")

✅ Training and evaluation functions defined


In [12]:
import time

print("="*70)
print("🚀 STARTING BERT TRAINING")
print("="*70)
print(f"   Model: DistilBERT")
print(f"   Epochs: {EPOCHS}")
print(f"   Batch size: {BATCH_SIZE}")
print(f"   Learning rate: {LEARNING_RATE}")
print(f"   Device: {device}")
print("="*70)

# Training history
history = {
    'train_loss': [],
    'train_accuracy': [],
    'val_loss': [],
    'val_accuracy': []
}

# Best model tracking
best_val_accuracy = 0
best_epoch = 0

# Training loop
for epoch in range(EPOCHS):
    print(f"\n{'='*50}")
    print(f"Epoch {epoch + 1}/{EPOCHS}")
    print(f"{'='*50}")
    
    start_time = time.time()
    
    # Train
    train_loss, train_acc = train_epoch(
        model, train_loader, optimizer, scheduler, device
    )
    
    # Validate
    val_loss, val_acc, val_preds, val_labels, val_probs = evaluate(
        model, val_loader, device
    )
    
    epoch_time = time.time() - start_time
    
    # Save history
    history['train_loss'].append(train_loss)
    history['train_accuracy'].append(train_acc)
    history['val_loss'].append(val_loss)
    history['val_accuracy'].append(val_acc)
    
    # Print metrics
    print(f"\n📊 Epoch {epoch+1} Results:")
    print(f"   Train Loss: {train_loss:.4f} | Train Acc: {train_acc*100:.2f}%")
    print(f"   Val Loss:   {val_loss:.4f} | Val Acc:   {val_acc*100:.2f}%")
    print(f"   Time: {epoch_time:.1f}s")
    
    # Save best model
    if val_acc > best_val_accuracy:
        best_val_accuracy = val_acc
        best_epoch = epoch + 1
        
        # Save model
        model_save_path = project_dir / 'models' / 'best_model.pt'
        torch.save({
            'epoch': epoch + 1,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'val_accuracy': val_acc,
            'val_loss': val_loss
        }, model_save_path)
        
        print(f"   💾 Best model saved! Val Acc: {val_acc*100:.2f}%")

print("\n" + "="*70)
print("✅ TRAINING COMPLETE!")
print("="*70)
print(f"   Best epoch: {best_epoch}")
print(f"   Best validation accuracy: {best_val_accuracy*100:.2f}%")
print(f"   Improvement over TextBlob (68%): +{(best_val_accuracy - 0.68)*100:.2f}%")
print("="*70)

# Save history
history_df = pd.DataFrame(history)
history_df.to_csv(project_dir / 'results' / 'metrics' / 'training_history.csv', index=False)
print(f"\n✅ Training history saved")

🚀 STARTING BERT TRAINING
   Model: DistilBERT
   Epochs: 3
   Batch size: 16
   Learning rate: 2e-05
   Device: cpu

Epoch 1/3


Training:  97%|█████████▋| 514/532 [3:33:44<07:29, 24.95s/it]     


KeyboardInterrupt: 

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

epochs_range = range(1, EPOCHS + 1)

# Loss
ax1 = axes[0]
ax1.plot(epochs_range, history['train_loss'], 'b-o', label='Train Loss', linewidth=2, markersize=8)
ax1.plot(epochs_range, history['val_loss'], 'r-o', label='Val Loss', linewidth=2, markersize=8)
ax1.set_title('Training & Validation Loss', fontweight='bold', fontsize=14)
ax1.set_xlabel('Epoch', fontweight='bold')
ax1.set_ylabel('Loss', fontweight='bold')
ax1.legend(fontsize=12)
ax1.grid(True, alpha=0.3)
ax1.set_xticks(epochs_range)

# Accuracy
ax2 = axes[1]
ax2.plot(epochs_range, [acc*100 for acc in history['train_accuracy']], 
         'b-o', label='Train Accuracy', linewidth=2, markersize=8)
ax2.plot(epochs_range, [acc*100 for acc in history['val_accuracy']], 
         'r-o', label='Val Accuracy', linewidth=2, markersize=8)
ax2.axhline(y=68, color='gray', linestyle='--', linewidth=2, label='TextBlob Baseline (68%)')
ax2.set_title('Training & Validation Accuracy', fontweight='bold', fontsize=14)
ax2.set_xlabel('Epoch', fontweight='bold')
ax2.set_ylabel('Accuracy (%)', fontweight='bold')
ax2.legend(fontsize=12)
ax2.grid(True, alpha=0.3)
ax2.set_xticks(epochs_range)

plt.tight_layout()
save_path = project_dir / 'results' / 'plots' / 'training_history.png'
plt.savefig(str(save_path), dpi=300, bbox_inches='tight')
plt.show()

print(f"✅ Training history saved: {save_path}")

In [ ]:
print("🧪 Evaluating on test set...")

# Load best model
best_model_path = project_dir / 'models' / 'best_model.pt'
checkpoint = torch.load(str(best_model_path), map_location=device)
model.load_state_dict(checkpoint['model_state_dict'])

print(f"✅ Best model loaded from epoch {checkpoint['epoch']}")
print(f"   Val Accuracy: {checkpoint['val_accuracy']*100:.2f}%")

# Evaluate on test set
test_loss, test_acc, test_preds, test_labels, test_probs = evaluate(
    model, test_loader, device
)

print("\n" + "="*70)
print("📊 TEST SET RESULTS - DISTILBERT")
print("="*70)
print(f"   Test Loss: {test_loss:.4f}")
print(f"   Test Accuracy: {test_acc*100:.2f}%")
print(f"   TextBlob Baseline: 68.00%")
print(f"   Improvement: +{(test_acc - 0.68)*100:.2f}%")
print("="*70)

# Detailed classification report
print("\n📋 Classification Report:")
print(classification_report(
    test_labels,
    test_preds,
    target_names=['Negative', 'Positive'],
    digits=4
))

# Save results
test_results = {
    'test_loss': float(test_loss),
    'test_accuracy': float(test_acc),
    'textblob_baseline': 0.68,
    'improvement': float(test_acc - 0.68)
}
with open(project_dir / 'results' / 'metrics' / 'test_results.json', 'w') as f:
    json.dump(test_results, f, indent=4)

In [ ]:
from sklearn.metrics import confusion_matrix
import seaborn as sns

# Confusion matrix
cm = confusion_matrix(test_labels, test_preds)

plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Negative', 'Positive'],
            yticklabels=['Negative', 'Positive'],
            cbar_kws={'label': 'Count'})

# Add percentages
for i in range(2):
    for j in range(2):
        pct = cm[i, j] / cm[i].sum() * 100
        plt.text(j+0.5, i+0.75, f'({pct:.1f}%)',
                ha='center', va='center',
                color='red' if i != j else 'darkgreen',
                fontweight='bold', fontsize=11)

plt.title(f'Confusion Matrix - DistilBERT\nAccuracy: {test_acc*100:.2f}%',
         fontweight='bold', fontsize=14)
plt.ylabel('True Label', fontweight='bold', fontsize=12)
plt.xlabel('Predicted Label', fontweight='bold', fontsize=12)
plt.tight_layout()

save_path = project_dir / 'results' / 'plots' / 'confusion_matrix.png'
plt.savefig(str(save_path), dpi=300, bbox_inches='tight')
plt.show()

print(f"✅ Confusion matrix saved: {save_path}")
print(f"\n📊 Confusion Matrix Details:")
print(f"   TN (Negative → Negative): {cm[0,0]}")
print(f"   FP (Negative → Positive): {cm[0,1]}")
print(f"   FN (Positive → Negative): {cm[1,0]}")
print(f"   TP (Positive → Positive): {cm[1,1]}")

In [ ]:
from sklearn.metrics import roc_curve, auc

# ROC Curve
test_probs_array = np.array(test_probs)
fpr, tpr, thresholds = roc_curve(test_labels, test_probs_array[:, 1])
roc_auc = auc(fpr, tpr)

plt.figure(figsize=(10, 8))
plt.plot(fpr, tpr, color='darkorange', lw=3,
         label=f'DistilBERT (AUC = {roc_auc:.4f})')
plt.plot([0, 1], [0, 1], color='navy', lw=2,
         linestyle='--', label='Random Classifier')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate', fontweight='bold', fontsize=12)
plt.ylabel('True Positive Rate', fontweight='bold', fontsize=12)
plt.title('ROC Curve - DistilBERT Sentiment Analysis',
         fontweight='bold', fontsize=14)
plt.legend(loc="lower right", fontsize=12)
plt.grid(True, alpha=0.3)
plt.tight_layout()

save_path = project_dir / 'results' / 'plots' / 'roc_curve.png'
plt.savefig(str(save_path), dpi=300, bbox_inches='tight')
plt.show()

print(f"✅ ROC curve saved: {save_path}")
print(f"📈 AUC Score: {roc_auc:.4f}")

In [ ]:
# Model comparison visualization
fig, ax = plt.subplots(figsize=(12, 6))

models = ['TextBlob\n(Baseline)', 'DistilBERT\n(Fine-tuned)']
accuracies = [68.0, test_acc * 100]
colors = ['#e74c3c', '#2ecc71']

bars = ax.bar(models, accuracies, color=colors, edgecolor='black',
             linewidth=1.5, width=0.4)

# Add value labels
for bar, acc in zip(bars, accuracies):
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.5,
           f'{acc:.2f}%', ha='center', va='bottom',
           fontweight='bold', fontsize=14)

# Add improvement arrow
ax.annotate(f'+{(test_acc*100 - 68):.2f}% improvement!',
           xy=(1, test_acc * 100),
           xytext=(0.5, (68 + test_acc * 100) / 2),
           fontsize=12, fontweight='bold', color='darkblue',
           arrowprops=dict(arrowstyle='->', color='darkblue', lw=2),
           ha='center')

ax.set_title('Model Comparison: TextBlob vs DistilBERT',
            fontweight='bold', fontsize=14)
ax.set_ylabel('Accuracy (%)', fontweight='bold', fontsize=12)
ax.set_ylim(50, 100)
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
save_path = project_dir / 'results' / 'plots' / 'model_comparison.png'
plt.savefig(str(save_path), dpi=300, bbox_inches='tight')
plt.show()

print(f"✅ Model comparison saved: {save_path}")

In [ ]:
from transformers import DistilBertTokenizer

# Save tokenizer for deployment
tokenizer_path = project_dir / 'models' / 'tokenizer'
tokenizer_path.mkdir(exist_ok=True)
tokenizer.save_pretrained(str(tokenizer_path))

print(f"✅ Tokenizer saved: {tokenizer_path}")

# Save model config
model_config = {
    'model_name': MODEL_NAME,
    'num_labels': 2,
    'max_length': MAX_LENGTH,
    'classes': ['Negative', 'Positive'],
    'test_accuracy': float(test_acc),
    'roc_auc': float(roc_auc),
    'best_epoch': int(best_epoch),
    'textblob_baseline': 0.68,
    'improvement': float(test_acc - 0.68)
}

config_path = project_dir / 'models' / 'model_config.json'
with open(config_path, 'w') as f:
    json.dump(model_config, f, indent=4)

print(f"✅ Model config saved: {config_path}")

In [ ]:
def predict_sentiment(text, model, tokenizer, device, max_length=256):
    """
    Predict sentiment for a custom text
    """
    # Tokenize
    encoding = tokenizer.encode_plus(
        text,
        add_special_tokens=True,
        max_length=max_length,
        padding='max_length',
        truncation=True,
        return_attention_mask=True,
        return_tensors='pt'
    )
    
    # Move to device
    input_ids = encoding['input_ids'].to(device)
    attention_mask = encoding['attention_mask'].to(device)
    
    # Predict
    model.eval()
    with torch.no_grad():
        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )
        logits = outputs.logits
        probs = torch.softmax(logits, dim=1)
        pred = torch.argmax(logits, dim=1)
    
    sentiment = 'POSITIVE' if pred.item() == 1 else 'NEGATIVE'
    confidence = probs[0][pred.item()].item()
    
    return {
        'text': text[:100] + '...' if len(text) > 100 else text,
        'sentiment': sentiment,
        'confidence': confidence,
        'positive_prob': probs[0][1].item(),
        'negative_prob': probs[0][0].item()
    }

# Test with sample reviews
test_reviews = [
    "This product is absolutely amazing! Best purchase I've ever made!",
    "Terrible quality, broke after 2 days. Complete waste of money.",
    "It's okay, nothing special but gets the job done.",
    "I love this so much! Exceeded all my expectations!",
    "Very disappointed. The description was completely misleading.",
    "Great value for money. Would definitely recommend!"
]

print("🔍 Testing model on custom reviews:")
print("="*70)

results = []
for review in test_reviews:
    result = predict_sentiment(review, model, tokenizer, device)
    results.append(result)
    
    emoji = "✅" if result['sentiment'] == 'POSITIVE' else "❌"
    print(f"\n{emoji} Review: {result['text']}")
    print(f"   Sentiment: {result['sentiment']}")
    print(f"   Confidence: {result['confidence']*100:.1f}%")
    print(f"   Positive: {result['positive_prob']*100:.1f}% | Negative: {result['negative_prob']*100:.1f}%")

print("="*70)

In [ ]:
print("\n" + "="*70)
print("🎉 WEEK 2 - BERT MODEL TRAINING COMPLETE!")
print("="*70)

print("\n✅ Completed Tasks:")
print("   ✓ DistilBERT tokenizer loaded")
print("   ✓ Custom PyTorch Dataset created")
print("   ✓ DataLoaders configured")
print("   ✓ DistilBERT model loaded and fine-tuned")
print("   ✓ Training with learning rate warmup")
print("   ✓ Best model saved")
print("   ✓ Evaluated on test set")
print("   ✓ Confusion matrix and ROC curve generated")
print("   ✓ Model comparison visualized")
print("   ✓ Custom review prediction tested")

print(f"\n📊 Model Performance:")
print(f"   TextBlob Baseline: 68.00%")
print(f"   DistilBERT: {test_acc*100:.2f}%")
print(f"   Improvement: +{(test_acc - 0.68)*100:.2f}%")
print(f"   AUC Score: {roc_auc:.4f}")

print(f"\n📁 Files Created:")
print(f"   • models/best_model.pt")
print(f"   • models/tokenizer/")
print(f"   • models/model_config.json")
print(f"   • results/plots/training_history.png")
print(f"   • results/plots/confusion_matrix.png")
print(f"   • results/plots/roc_curve.png")
print(f"   • results/plots/model_comparison.png")
print(f"   • results/metrics/training_history.csv")
print(f"   • results/metrics/test_results.json")

print(f"\n🎯 Next Steps (Week 3):")
print("   1. Aspect-based sentiment extraction")
print("   2. Multi-class sentiment (5 star rating)")
print("   3. Model optimization")
print("   4. Error analysis")
print("   5. Prepare for deployment")

print("\n💡 Ready for Week 3?")
print("   Reply: '✅ Week 2 done! BERT accuracy: XX%'")
print("="*70)